In [4]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
CACHE_PATH = PROJECT_ROOT / "cache"
CACHE_PATH.mkdir(parents=True, exist_ok=True)
sys.path.insert(0, str(PROJECT_ROOT))

# import pyfredapi as pf
import pandas as pd
# from fred_api_key import FRED_API_KEY
from time import sleep

# API_KEY = FRED_API_KEY

START_DATE = "2000-01-01"
END_DATE = "2026-06-30"

In [ ]:
# Helper functions to obtain FOMC meeting information
# No need to run if FOMC is already cached

import re
from datetime import datetime, timedelta

# import pandas as pd
import requests
from bs4 import BeautifulSoup


MONTHS = {
    "January": 1,
    "February": 2,
    "March": 3,
    "April": 4,
    "May": 5,
    "June": 6,
    "July": 7,
    "August": 8,
    "September": 9,
    "October": 10,
    "November": 11,
    "December": 12,

    # abbreviations
    "Jan": 1,
    "Feb": 2,
    "Mar": 3,
    "Apr": 4,
    "Jun": 6,
    "Jul": 7,
    "Aug": 8,
    "Sep": 9,
    "Sept": 9,
    "Oct": 10,
    "Nov": 11,
    "Dec": 12,
}


def parse_meeting_title(title: str, year: int):
    """
    Parse a meeting title into output rows.

    Examples
    --------
    March 21
    January 30-31
    July 31-August 1
    Jul/Aug 31-1

    Returns
    -------
    list[dict]
    """

    title = title.strip()

    # remove parenthetical notes
    title = re.sub(r"\s*\([^)]*\)", "", title).strip()

    # Case 1
    # March 21
    m = re.fullmatch(r"([A-Za-z]+)\s+(\d{1,2})", title)

    if m:

        month = MONTHS[m.group(1)]
        day = int(m.group(2))

        return [{
            "date": datetime(year, month, day).strftime("%Y-%m-%d"),
            "is_fomc_day": 1,
            "is_fomc_press_conference": 1
        }]

    # Case 2
    # January 30-31
    m = re.fullmatch(
        r"([A-Za-z]+)\s+(\d{1,2})-(\d{1,2})",
        title
    )

    if m:

        month = MONTHS[m.group(1)]

        first = datetime(
            year,
            month,
            int(m.group(2))
        )

        second = datetime(
            year,
            month,
            int(m.group(3))
        )

        return [
            {
                "date": first.strftime("%Y-%m-%d"),
                "is_fomc_day": 0,
                "is_fomc_press_conference": 1
            },
            {
                "date": second.strftime("%Y-%m-%d"),
                "is_fomc_day": 1,
                "is_fomc_press_conference": 0
            }
        ]

    # Case 3
    # July 31-August 1
    m = re.fullmatch(
        r"([A-Za-z]+)\s+(\d{1,2})-([A-Za-z]+)\s+(\d{1,2})",
        title
    )

    if m:

        first = datetime(
            year,
            MONTHS[m.group(1)],
            int(m.group(2))
        )

        second = datetime(
            year,
            MONTHS[m.group(3)],
            int(m.group(4))
        )

        return [
            {
                "date": first.strftime("%Y-%m-%d"),
                "is_fomc_day": 0,
                "is_fomc_press_conference": 1
            },
            {
                "date": second.strftime("%Y-%m-%d"),
                "is_fomc_day": 1,
                "is_fomc_press_conference": 0
            }
        ]

    # Case 4
    # Jul/Aug 31-1
    m = re.fullmatch(
        r"([A-Za-z]+)/([A-Za-z]+)\s+(\d{1,2})-(\d{1,2})",
        title
    )

    if m:

        first = datetime(
            year,
            MONTHS[m.group(1)],
            int(m.group(3))
        )

        second = datetime(
            year,
            MONTHS[m.group(2)],
            int(m.group(4))
        )

        return [
            {
                "date": first.strftime("%Y-%m-%d"),
                "is_fomc_day": 0,
                "is_fomc_press_conference": 1
            },
            {
                "date": second.strftime("%Y-%m-%d"),
                "is_fomc_day": 1,
                "is_fomc_press_conference": 0
            }
        ]

    # raise ValueError(f"Unknown meeting title: {title}")
    return None


def scrape_fomc_historical_year(year: int) -> pd.DataFrame:
    """
    Historical pages (2000-2020)
    """

    url = f"https://www.federalreserve.gov/monetarypolicy/fomchistorical{year}.htm"

    r = requests.get(url)
    r.raise_for_status()

    soup = BeautifulSoup(r.text, "html.parser")

    article = soup.find("div", id="article")

    rows = []

    for panel in article.select("div.panel.panel-default"):

        h5 = panel.find("h5")

        if h5 is None:
            continue

        heading = h5.get_text(" ", strip=True)

        # require Statement
        has_statement = any(
            "statement" in a.get_text(strip=True).lower()
            for a in panel.find_all("a")
        )

        if not has_statement:
            continue

        # skip notation votes
        if "notation vote" in heading.lower():
            continue

        title = re.sub(r"\s*-\s*\d{4}$", "", heading)
        title = title.replace(" Meeting", "")
        title = title.strip()

        parsed = parse_meeting_title(title, year)

        if parsed:
            rows.extend(parsed)
        else:
            print(f"Skipping unknown format: {heading}")

    df = pd.DataFrame(rows)

    if df.empty:
        return pd.DataFrame(
            columns=[
                "date",
                "is_fomc_day",
                "is_fomc_press_conference"
            ]
        )

    return (
        df
        .drop_duplicates()
        .sort_values("date")
        .reset_index(drop=True)
    )


def scrape_fomc_calendar_all() -> pd.DataFrame:
    """
    Scrape all completed meetings (2021+) from the current FOMC calendar page.
    """

    url = "https://www.federalreserve.gov/monetarypolicy/fomccalendars.htm"

    r = requests.get(url)
    r.raise_for_status()

    soup = BeautifulSoup(r.text, "html.parser")

    rows = []

    for panel in soup.select("div.panel.panel-default"):

        h4 = panel.find("h4")

        if h4 is None:
            continue

        heading = h4.get_text(" ", strip=True)

        # Skip anything that isn't a yearly FOMC panel
        m = re.match(r"(\d{4}) FOMC Meetings", heading)

        if m is None:
            continue

        for meeting in panel.select("div.fomc-meeting"):

            # Find the policy statement link.
            # We identify it by its href, not its displayed text.
            statement_link = meeting.find(
                "a",
                href=re.compile(r"monetary\d{8}a")
            )

            if statement_link is None:
                # Future meetings (or malformed entries)
                continue

            # Decision date comes from Statement URL.
            href = statement_link.get("href", "")

            m = re.search(r"(\d{8})", href)

            if m is None:
                continue

            decision = datetime.strptime(
                m.group(1),
                "%Y%m%d"
            )

            # Was this one day or two?
            date_text = (
                meeting
                .select_one(".fomc-meeting__date")
                .get_text(strip=True)
                .replace("*", "")
            )

            if "-" in date_text:

                conference = decision - timedelta(days=1)

                rows.append({
                    "date": conference.strftime("%Y-%m-%d"),
                    "is_fomc_day": 0,
                    "is_fomc_press_conference": 1
                })

                rows.append({
                    "date": decision.strftime("%Y-%m-%d"),
                    "is_fomc_day": 1,
                    "is_fomc_press_conference": 0
                })

            else:

                rows.append({
                    "date": decision.strftime("%Y-%m-%d"),
                    "is_fomc_day": 1,
                    "is_fomc_press_conference": 1
                })

    df = pd.DataFrame(rows)

    return (
        df
        .drop_duplicates()
        .sort_values("date")
        .reset_index(drop=True)
    )

In [5]:
# Executing the helper functions to obtain FOMC meeting information
# No need to run if FOMC is already cached

# Historical pages (2000-2020)
dfs = []

for year in range(2000, 2021):
    print(f"Scraping historical {year}...")
    dfs.append(scrape_fomc_historical_year(year))
    sleep(0.5)

fomc_historical = (
    pd.concat(dfs, ignore_index=True)
      .sort_values("date")
      .reset_index(drop=True)
)

# Calendar page (2021-present)
print("Scraping calendar page (2021-present)...")

fomc_recent = scrape_fomc_calendar_all()

# Combine
fomc = (
    pd.concat(
        [fomc_historical, fomc_recent],
        ignore_index=True
    )
    .drop_duplicates()
    .sort_values("date")
    .reset_index(drop=True)
)

# Save
fomc.to_csv(CACHE_PATH / "fomc_calendar_2000_present.csv", index=False)

print(fomc.head())
print(fomc.tail())
print(f"Total FOMC dates scraped: {len(fomc)}")

Scraping historical 2000...
Scraping historical 2001...
Skipping unknown format: January 3 Conference Call - 2001
Skipping unknown format: April 18 Conference Call - 2001
Skipping unknown format: September 17 Conference Call - 2001
Scraping historical 2002...
Scraping historical 2003...
Scraping historical 2004...
Scraping historical 2005...
Scraping historical 2006...
Scraping historical 2007...
Skipping unknown format: August 10 Conference Call - 2007
Skipping unknown format: August 16 Conference Call - 2007
Scraping historical 2008...
Skipping unknown format: January 21 Conference Call - 2008
Skipping unknown format: March 10 Conference Call - 2008
Skipping unknown format: October 7 Conference Call - 2008
Scraping historical 2009...
Scraping historical 2010...
Skipping unknown format: May 9 Conference Call - 2010
Scraping historical 2011...
Scraping historical 2012...
Scraping historical 2013...
Scraping historical 2014...
Scraping historical 2015...
Scraping historical 2016...
Scra